[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C59_VLA_Perception_Interface_Course/03_perception_interface/03_perception_interface.ipynb)

# 03 · TSR 输出如何进入 VLA（融合层次 / schema / 校准 / 错误注入 / prompt 稳定性）

目标：把「感知输出接口」从一句口号变成**可运行、可断言、可上线**的东西。
这是整门课的核心，也是 JD 里「设计感知输出接口以支持 VLA 模型」那条职责的直接对应。

本 notebook 你会亲手实现：
1. **三种融合层次的 token 与信息量账**（bits/token 效率差 1000 倍，以及交叉点在哪）
2. **完整的 TSR→VLA 序列化 schema**：渲染 + 解析 + **round-trip 测试** + 字段消融
3. **针孔距离估计与它的不确定度**：为什么 82 米这个数不带 σ 就是在撒谎
4. **置信度校准**：可靠性图、ECE、温度缩放，以及**期望代价决策在 ECE 最小处取到最优**的验证
5. **感知错误注入实验**：下游动作错误率如何随「置信度是否传递 / 是否校准」变化
6. **非对称门控**：把代价不对称写进规则，超速率降 72%
7. **prompt 模板的三种静默故障**：位置错位 / 哨兵值被吃掉 / 格式漂移

> 心智模型：**接口丢掉的信息，下游永远补不回来。
> 不传置信度不是少传一个字段，是永久删除了下游做风险决策的能力。**

## 1 · 三种融合层次的 token 与信息量账

先把「符号级贵不贵」这件事算清楚，而不是靠感觉。

In [ ]:
import numpy as np, math, json, re
rng = np.random.default_rng(0)

token_est = lambda s: len(s) // 4 + 1        # 4 字符/token 的经验法则（够用即可）

# 一份完整 schema 的字段位预算（下游真正需要的信息量，单位 bit）
BIT_BUDGET = {
    'track_id': 12,      # 4096 个并发 track
    'cls': 8,            # ~200 个标志类别
    'value_kph': 8,      # 限速值
    'conf_det': 7, 'conf_cls': 7,       # 1/128 分辨率足够
    'x_fwd_m': 11,       # 0~200 m @ 0.1 m
    'y_left_m': 9,       # ±20 m @ 0.1 m
    'sigma_dist_m': 7,
    'lane_assoc': 3, 'n_obs': 5, 'state': 2,
}
SYMBOLIC_BITS = sum(BIT_BUDGET.values())
SYMBOLIC_TOKENS_PER_OBJ = 39     # 紧凑键值行的实测 token 数（第 5 节会算出来）
SLIM_TOKENS_PER_OBJ = 15         # 只留决策必需字段的精简版
FEATURE_DIM, FEATURE_BITS_PER_DIM = 256, 8
BEV_QUERIES = 100                # 与目标数**无关**的固定开销
HEADER_TOKENS = 30               # schema_version + 自车状态 + 表头

print(f'符号级：{SYMBOLIC_BITS} bit/目标，{SYMBOLIC_TOKENS_PER_OBJ} token/目标'
      f'  → {SYMBOLIC_BITS/SYMBOLIC_TOKENS_PER_OBJ:.2f} bit/token')
feat_bits = FEATURE_DIM * FEATURE_BITS_PER_DIM
print(f'特征级：{feat_bits} bit/目标，1 token/目标'
      f'  → {feat_bits:.0f} bit/token')
ratio = feat_bits / (SYMBOLIC_BITS / SYMBOLIC_TOKENS_PER_OBJ)
print(f'\n★ 特征级的 **bit/token 效率是符号级的 {ratio:.0f} 倍**')
assert ratio > 500
print('   —— 但那 2048 个 bit **不可读、不可审计、不能写进事故报告**。')
print('   符号级付出 1000 倍的 token 效率代价，换的不是信息量，是**可解释性**。')

In [ ]:
def token_cost(n_obj, scheme):
    if scheme == 'symbolic_full':
        return HEADER_TOKENS + SYMBOLIC_TOKENS_PER_OBJ * n_obj
    if scheme == 'symbolic_slim':
        return HEADER_TOKENS + SLIM_TOKENS_PER_OBJ * n_obj
    if scheme == 'feature':
        return n_obj
    if scheme == 'bev_query':
        return BEV_QUERIES
    raise ValueError(scheme)

SCENES = [('高速 · 空旷', 1), ('高速 · 龙门架', 3), ('城市 · 普通路口', 8),
          ('城市 · 复杂路口', 20), ('城市 · 龙门架+商铺招牌', 40)]
print(f"{'场景':<24s} {'目标数':>6s} {'符号(全)':>9s} {'符号(精简)':>11s} {'特征级':>7s} {'BEV query':>10s}")
for name, n in SCENES:
    print(f'{name:<24s} {n:>6d} {token_cost(n,"symbolic_full"):>9d} '
          f'{token_cost(n,"symbolic_slim"):>11d} {token_cost(n,"feature"):>7d} '
          f'{token_cost(n,"bev_query"):>10d}')

def crossover(scheme_a, scheme_b, n_max=200):
    """找到 a 开始比 b 贵的最小目标数。"""
    for n in range(1, n_max):
        if token_cost(n, scheme_a) > token_cost(n, scheme_b):
            return n
    return None

n_cross = crossover('symbolic_full', 'bev_query')
n_cross_slim = crossover('symbolic_slim', 'bev_query')
print(f'\n符号(全) 从 {n_cross} 个目标起就比固定 {BEV_QUERIES} 个 BEV query 贵')
print(f'符号(精简) 从 {n_cross_slim} 个目标起才反超')
assert n_cross == 2 and n_cross_slim == 5
print(f'\n⚠️  40 个目标的城市场景：符号级要 {token_cost(40,"symbolic_full")} token 光描述场景，')
print(f'    而 BEV query 永远是 {BEV_QUERIES} —— 这就是量产走中间表示级的第一个理由。')
print('✅ 但判据不是 token：**「这个信息出错以后需不需要事后说清楚」——需要就必须走符号级**。')
print('   务实形态是混合：稠密几何走 BEV，少量安全强语义（限速值/禁行）额外走符号通路。')

## 2 · TSR → VLA 的序列化 schema：渲染、解析、round-trip

**渲染与解析必须成对实现，并做 round-trip 测试**——这一个测试能挡住八成模板事故。

In [ ]:
SCHEMA_VERSION = 'tsr/1.3'
FIELD_ORDER = ['track_id', 'cls', 'value_kph', 'conf_det', 'conf_cls',
               'x_fwd_m', 'y_left_m', 'sigma_dist_m', 'lane_assoc', 'n_obs', 'state']
NUM_FMT = {'track_id': '{:d}', 'value_kph': '{:d}', 'n_obs': '{:d}',
           'conf_det': '{:.2f}', 'conf_cls': '{:.2f}',
           'x_fwd_m': '{:.1f}', 'y_left_m': '{:.1f}', 'sigma_dist_m': '{:.1f}'}
LANE_PRIORITY = {'ego': 0, 'left': 1, 'right': 1, 'opposite': 2, 'service': 2, 'unknown': 3}
TAIL_TOKENS = 8            # 为 prompt 尾部的 truncated 标记预留
INT_FIELDS = {'track_id', 'value_kph', 'n_obs'}
FLOAT_FIELDS = {'conf_det', 'conf_cls', 'x_fwd_m', 'y_left_m', 'sigma_dist_m'}
NULL = 'null'

def fmt_value(k, v, null=NULL):
    if v is None:
        return null
    f = NUM_FMT.get(k)
    return f.format(v) if f else str(v)

def render_line(det, order=FIELD_ORDER, keyed=True, null=NULL):
    parts = [(f'{k}={fmt_value(k, det.get(k), null)}' if keyed
              else fmt_value(k, det.get(k), null)) for k in order]
    return '|'.join(parts)

def parse_keyed(line, strict=True):
    out = {}
    for p in line.split('|'):
        k, sep, v = p.partition('=')
        if strict and not sep:
            raise ValueError(f'非键值片段: {p!r}')      # **fail loud**，绝不「尽力猜」
        out[k] = None if v == NULL else v
    # 类型还原
    for k in list(out):
        if out[k] is None:
            continue
        if k in INT_FIELDS:
            out[k] = int(out[k])
        elif k in FLOAT_FIELDS:
            out[k] = float(out[k])
    return out

def parse_positional(line, order=FIELD_ORDER):
    vals = line.split('|')
    out = {k: (None if v == NULL else v) for k, v in zip(order, vals)}
    return out

DET = dict(track_id=4127, cls='speed_limit', value_kph=60, conf_det=0.93, conf_cls=0.71,
           x_fwd_m=82.4, y_left_m=-3.1, sigma_dist_m=6.9, lane_assoc='ego',
           n_obs=3, state='tentative')
line = render_line(DET)
print(line)
print(f'\n≈ {token_est(line)} token（按 4 字符/token 的经验法则）')

# ★ round-trip 测试：渲染 -> 解析 -> 逐字段相等
back = parse_keyed(line)
for k in FIELD_ORDER:
    assert back[k] == DET[k], (k, back[k], DET[k])
assert token_est(line) == SYMBOLIC_TOKENS_PER_OBJ
print('✅ round-trip 通过：11 个字段全部逐一相等，token 数与第 1 节的假设一致')

In [ ]:
# 字段消融：拿掉某个字段，下游哪条判断就做不了了
def downstream_capabilities(det):
    """给定一条（可能缺字段的）检测，返回下游还能做的判断。"""
    caps = {}
    caps['知道是什么'] = det.get('cls') is not None
    caps['知道限速值'] = det.get('value_kph') is not None
    caps['知道还有多远'] = det.get('x_fwd_m') is not None
    caps['能算安全裕度'] = det.get('sigma_dist_m') is not None and caps['知道还有多远']
    caps['知道管不管我'] = det.get('lane_assoc') is not None
    caps['能跨帧确认'] = det.get('track_id') is not None
    caps['能区分单帧强证据/多帧弱证据'] = (det.get('n_obs') is not None
                                          and det.get('conf_cls') is not None)
    caps['能做风险决策'] = det.get('conf_cls') is not None
    return caps

FULL = downstream_capabilities(DET)
print(f"{'拿掉的字段':<16s} {'失去的下游能力'}")
lost_map = {}
for k in ['track_id', 'value_kph', 'conf_cls', 'sigma_dist_m', 'lane_assoc', 'n_obs']:
    d = {kk: vv for kk, vv in DET.items() if kk != k}
    c = downstream_capabilities(d)
    lost = [cap for cap in FULL if FULL[cap] and not c[cap]]
    lost_map[k] = lost
    print(f'{k:<16s} {"、".join(lost)}')

assert lost_map['track_id'] == ['能跨帧确认']
assert '能做风险决策' in lost_map['conf_cls'] and '能区分单帧强证据/多帧弱证据' in lost_map['conf_cls']
assert lost_map['sigma_dist_m'] == ['能算安全裕度']
assert lost_map['lane_assoc'] == ['知道管不管我']
print('\n✅ **每个字段都必须能回答「拿掉它，下游哪条判断就做不了了」** ——')
print('   答不上来的字段就是噪声：占 token、让模板变脆、还要跟着版本走。')
print('⚠️  注意 conf_cls 一拿掉，同时丢了两项能力（风险决策 + 时序/单帧的区分），')
print('    这就是第 4 节说的「置信度不是一个字段，是一整类能力的入口」。')

## 3 · 位置怎么传：针孔距离与它的不确定度

$f_{px} = \dfrac{W_{px}}{2\tan(\mathrm{FOV}_h/2)}$，　$Z = \dfrac{f_{px}S}{w_{px}}$，　$\dfrac{\mathrm{d}Z}{Z} = -\dfrac{\mathrm{d}w}{w}$

In [ ]:
W_PX, FOV_H_DEG = 1920, 60.0
SIGN_SIZE_M = 0.6                       # 中国圆形限速牌常见直径
F_PX = W_PX / (2 * math.tan(math.radians(FOV_H_DEG) / 2))
print(f'焦距 f_px = {W_PX} / (2·tan({FOV_H_DEG/2}°)) = {F_PX:.1f} px')

def distance_from_width(w_px, S=SIGN_SIZE_M, f=F_PX):
    return f * S / w_px

def sigma_distance(w_px, sigma_w_px=1.0, S=SIGN_SIZE_M, f=F_PX):
    """一阶传播：σ_Z = (Z/w)·σ_w = (Z²/(f·S))·σ_w  —— **误差随距离平方增长**"""
    Z = distance_from_width(w_px, S, f)
    return Z / w_px * sigma_w_px

print(f"\n{'框宽 w':>8s} {'距离 Z':>9s} {'相对误差':>9s} {'σ_Z':>9s} {'±1σ 区间':>18s}")
for w in [60, 24, 12, 8]:
    Z, s = distance_from_width(w), sigma_distance(w)
    print(f'{w:>7d}px {Z:>8.1f}m {1.0/w:>8.2%} {s:>8.2f}m {f"[{Z-s:.1f}, {Z+s:.1f}]":>18s}')

Z12, s12 = distance_from_width(12), sigma_distance(12)
assert abs(Z12 - 83.1) < 0.2 and abs(s12 - 6.9) < 0.1
# 平方关系校验：距离翻倍（框宽减半），σ 应变 4 倍
assert abs(sigma_distance(6) / sigma_distance(12) - 4.0) < 1e-9
print(f'\n⚠️  12 px 的框 → 83.1 m，但 1 px 的框宽误差就是 **±6.9 m**。')
print('    30 m/s 下这 14 米的区间宽度 ≈ 0.5 秒决策提前量 —— 足以决定「平缓减速」还是「急刹」。')
print('✅ **σ_Z ∝ Z²**：距离翻倍，不确定度变 4 倍。所以远处目标的距离必须带 σ 一起传。')

In [ ]:
# 物理尺寸判错是更大的系统性误差来源
print('同一个 12 px 的框，若把 0.8 m 的牌子当成 0.6 m：')
Z_06, Z_08 = distance_from_width(12, 0.6), distance_from_width(12, 0.8)
print(f'  S=0.6 m → {Z_06:.1f} m ;  S=0.8 m → {Z_08:.1f} m ;  '
      f'系统性偏差 {abs(Z_08-Z_06)/Z_08:.1%}（{abs(Z_08-Z_06):.1f} m）')
assert abs(abs(Z_08 - Z_06) / Z_08 - 0.25) < 1e-9
print('  ⚠️  这是 **系统性偏差**，不是随机噪声 —— 多帧平均消不掉它。')
print('  ✅ 所以 schema 里的 cls 不只是语义，它同时是距离估计的**尺寸先验**。')

# 图像坐标 -> 自车坐标（这一步必须由感知做，不能甩给 VLA）
CX = W_PX / 2
def image_to_ego(u_px, w_px, S=SIGN_SIZE_M, f=F_PX):
    Z = distance_from_width(w_px, S, f)
    x_fwd = Z
    y_left = -(u_px - CX) * Z / f          # 图像 u 向右为正 -> 自车 y 向左为正
    return x_fwd, y_left

print(f"\n{'像素 u':>8s} {'框宽':>6s} {'x_fwd':>9s} {'y_left':>9s} {'在哪边'}")
for u, w in [(960, 12), (1204, 12), (700, 12), (1204, 60)]:
    x, y = image_to_ego(u, w)
    side = '正前' if abs(y) < 1 else ('左侧' if y > 0 else '右侧')
    print(f'{u:>8d} {w:>5d}px {x:>8.1f}m {y:>8.1f}m {side}')

x_c, y_c = image_to_ego(960, 12)
assert abs(y_c) < 1e-9, '图像中心必须映射到 y_left = 0'
x_r, y_r = image_to_ego(1204, 12)
assert y_r < 0, 'u > cx 表示在图像右侧 -> 自车坐标 y_left 为负'
# **坐标系断言测试**：放一个明确在左边的目标，检查下游认为它在左边
x_l, y_l = image_to_ego(700, 12)
assert y_l > 0, '坐标系符号错误！这个十行的测试能省掉一次线上事故'
print('\n✅ 坐标系断言测试通过（左为正 / 右为负 / 中心为 0）。')
print('⚠️  这类符号错误**只在明显偏侧的目标上暴露**，正前方的牌子看不出来 ——')
print('    于是问题只在特定路段复现，查起来极痛苦。所以必须有这个断言。')

## 4 · 置信度校准：可靠性图、ECE，以及期望代价决策

合成规则：每条检测有一个隐藏难度 $d\sim\mathcal N(0,1)$，它**真正正确的概率**是
$q=\sigma(1.6-1.3d)$；是否正确按 $q$ 抽 Bernoulli。
报告出来的置信度是 $\mathrm{conf}=\sigma(\mathrm{logit}(q)/T)$：
$T=1$ 完全校准，$T<1$ 过自信，$T>1$ 欠自信。

In [ ]:
N_DET = 200_000
sigmoid = lambda z: 1.0 / (1.0 + np.exp(-z))
logit = lambda p: np.log(p / (1.0 - p))

_r = np.random.default_rng(0)
difficulty = _r.standard_normal(N_DET)
q_true = sigmoid(1.6 - 1.3 * difficulty)             # 该检测**真正**正确的概率
is_correct = _r.random(N_DET) < q_true

def reported_conf(T):
    """温度缩放：T=1 完全校准；T<1 过自信；T>1 欠自信。"""
    return sigmoid(logit(np.clip(q_true, 1e-6, 1 - 1e-6)) / T)

def ece(conf, correct, n_bins=15):
    e, edges = 0.0, np.linspace(0.0, 1.0, n_bins + 1)
    for i in range(n_bins):
        m = (conf >= edges[i]) & (conf < edges[i + 1] if i < n_bins - 1 else conf <= 1.0)
        if m.sum() == 0:
            continue
        e += m.mean() * abs(conf[m].mean() - correct[m].mean())
    return float(e)

print(f'基础正确率 = {is_correct.mean():.4f}  → **感知错误率 {1-is_correct.mean():.2%}**')
print(f'\n可靠性图（T=1，完全校准）：')
c1 = reported_conf(1.0)
edges = np.linspace(0, 1, 11)
print(f"{'置信区间':>12s} {'样本占比':>9s} {'平均置信':>9s} {'实际准确率':>11s} {'差':>8s}")
for i in range(10):
    m = (c1 >= edges[i]) & (c1 < edges[i + 1] if i < 9 else c1 <= 1.0)
    if m.sum() < 50:
        continue
    print(f'[{edges[i]:.1f},{edges[i+1]:.1f}) {m.mean():>9.1%} {c1[m].mean():>9.3f} '
          f'{is_correct[m].mean():>11.3f} {c1[m].mean()-is_correct[m].mean():>+8.3f}')
assert ece(c1, is_correct) < 0.01
print(f'\nECE(T=1) = {ece(c1, is_correct):.4f}  ← 每个箱里「说 0.8 就真的对 80%」')

In [ ]:
C_WRONG, C_REJ = 10.0, 1.5                  # 采纳一个错检测的代价 / 拒绝退保守的代价
TAU_BAYES = 1.0 - C_REJ / C_WRONG           # 贝叶斯最优阈值，**算出来的不是调出来的**
print(f'贝叶斯阈值 p* = 1 − C_rej/C_wrong = 1 − {C_REJ}/{C_WRONG} = {TAU_BAYES}')

def expected_cost(conf, tau):
    """用**真实** q 结算实际代价；决策却只能看 conf（这正是失准伤人的地方）。"""
    acc = conf >= tau
    return float(np.mean(acc * (1 - q_true) * C_WRONG + (~acc) * C_REJ))

no_conf_cost = float(np.mean((1 - q_true) * C_WRONG))     # 不传置信度 = 全部采纳
all_rej_cost = C_REJ                                       # 完全不用感知

print(f"\n{'温度 T':>7s} {'ECE':>8s} {'接受率':>8s} {'期望代价':>9s} {'相对最优':>9s}")
rows = []
for T in [0.2, 0.35, 0.5, 0.7, 1.0, 1.5, 2.0, 3.0]:
    c = reported_conf(T)
    cost_ = expected_cost(c, TAU_BAYES)
    rows.append((T, ece(c, is_correct), cost_, float((c >= TAU_BAYES).mean())))
best_cost = min(r[2] for r in rows)
for T, e, cost_, acc in rows:
    print(f'{T:>7.2f} {e:>8.4f} {acc:>8.1%} {cost_:>9.4f} {cost_/best_cost-1:>+8.1%}')
print(f'\n{"不传置信度（全部采纳）":<22s} {no_conf_cost:>9.4f} {no_conf_cost/best_cost-1:>+8.1%}')
print(f'{"完全忽略感知（全部拒绝）":<22s} {all_rej_cost:>9.4f} {all_rej_cost/best_cost-1:>+8.1%}')

best_T = min(rows, key=lambda r: r[2])[0]
best_ece_T = min(rows, key=lambda r: r[1])[0]
assert best_T == 1.0 and best_ece_T == 1.0, '期望代价与 ECE 应在同一个 T 取到最小'
assert no_conf_cost > best_cost * 1.9, '不传置信度的代价应接近最优的两倍'
assert all_rej_cost > best_cost
print('\n★ **期望代价恰好在 ECE 最小（完全校准）处取到最小值** —— 这不是巧合：')
print('  贝叶斯决策规则用真实概率时最优，用被扭曲的概率必然次优。')
print(f'⚠️  不传置信度的代价比最优高 {no_conf_cost/best_cost-1:.0%}，比「完全不用感知」还差得多但也没好到哪去。')

In [ ]:
# ★ 必须讲清楚的微妙点：单调失准 + 重调阈值 = 决策完全不变
def auc(score, label):
    order = np.argsort(score, kind='stable')
    ranks = np.empty(len(score)); ranks[order] = np.arange(1, len(score) + 1)
    npos = int(label.sum()); nneg = len(score) - npos
    return float((ranks[label].sum() - npos * (npos + 1) / 2) / (npos * nneg))

grid = np.linspace(0.01, 0.999, 400)
print(f"{'T':>6s} {'AUC(排序质量)':>14s} {'固定 τ*=0.85 的代价':>19s} {'重调最优阈值后':>15s}")
aucs, fixed, tuned = [], [], []
for T in [0.35, 0.5, 1.0, 2.0, 3.0]:
    c = reported_conf(T)
    a = auc(c, is_correct)
    f_ = expected_cost(c, TAU_BAYES)
    t_ = min(expected_cost(c, t) for t in grid)
    aucs.append(a); fixed.append(f_); tuned.append(t_)
    print(f'{T:>6.2f} {a:>14.6f} {f_:>19.4f} {t_:>15.4f}')

assert max(aucs) - min(aucs) < 1e-6, '单调变换不改变排序，AUC 必须完全相同'
assert max(tuned) - min(tuned) < 0.005, '重调阈值后可达代价几乎相同'
assert max(fixed) - min(fixed) > 0.3, '固定阈值下失准会显著变差'
print('\n⚠️  **AUC 完全相同、重调阈值后代价也相同** —— 这就是「调调阈值就好了，')
print('    所以校准没用」这个错误结论的来源。它在那个特定配置下确实「好了」。')
print('\n✅ 但这个补偿在三种情况下立刻塌掉：')
print('   ① 阈值是固定的 / 多个消费者共享 → 感知一升级，全线行为漂移，没人会想到去重调')
print('   ② 决策要用概率做**算术**（期望代价、风险预算、代价不对称门控）→ 系统性算错')
print('   ③ 多源融合（相机 p₁ × 地图先验 p₂）→ 两个失准概率相乘，误差放大且方向不可预测')
print('\n★ 所以：**校准不是为了改善排序（它做不到），是为了让「概率」这个数能被当作概率来算。**')

## 5 · 感知错误注入实验：下游动作错误率随接口设计怎么变

场景：每一帧的真实限速 $\in\{60,80,100,120\}$；感知按上面的 $q$ 给出可能错误的读数；
地图/上一段先验有 25% 的错误率（会因施工、限速调整而过时）。
下游要决定**应用哪个限速**，代价严重不对称：**超速 ×10，过度限速 ×3**。

In [ ]:
LIMITS = np.array([60, 80, 100, 120])
_r2 = np.random.default_rng(0)
i_true = _r2.integers(0, 4, N_DET)
L_true = LIMITS[i_true]
# 感知读数：正确时 = 真值；错误时随机错到别的档
off1 = _r2.integers(1, 4, N_DET)
L_det = np.where(is_correct, L_true, LIMITS[(i_true + off1) % 4])
# 地图 / 上一段先验：75% 正确
prior_ok = _r2.random(N_DET) < 0.75
off2 = _r2.integers(1, 4, N_DET)
L_prior = np.where(prior_ok, L_true, LIMITS[(i_true + off2) % 4])

print(f'感知错误率 {(L_det != L_true).mean():.2%}   先验错误率 {(L_prior != L_true).mean():.2%}')

W_OVER, W_UNDER = 10.0, 3.0        # 超速是安全问题；过度限速是效率/舒适问题
def action_risk(applied):
    over = float((applied > L_true).mean())          # 超速：危险且不可恢复
    under = float((applied < L_true).mean())         # 过度限速：低效但可恢复
    return over, under, W_OVER * over + W_UNDER * under

POLICIES = {
    'A 不传置信度（全采纳）': lambda c: L_det,
    'B 对称阈值 0.70':        lambda c: np.where(c >= 0.70, L_det, L_prior),
    'C 非对称门控':            lambda c: np.where(
        np.where(L_det > L_prior, c >= 0.90, c >= 0.30), L_det, L_prior),
    'D 完全忽略感知':          lambda c: L_prior,
}

for T, tag in [(1.0, '校准 T=1'), (0.3, '过自信 T=0.3'), (3.0, '欠自信 T=3')]:
    c = reported_conf(T)
    print(f'\n─── {tag}（ECE={ece(c, is_correct):.4f}）───')
    print(f"{'下游策略':<24s} {'超速率':>8s} {'过度限速率':>11s} {'加权风险':>9s}")
    for name, fn in POLICIES.items():
        o, u, r = action_risk(fn(c))
        print(f'{name:<24s} {o:>8.2%} {u:>11.2%} {r:>9.4f}')

In [ ]:
c_cal, c_over, c_under = reported_conf(1.0), reported_conf(0.3), reported_conf(3.0)
rA = action_risk(POLICIES['A 不传置信度（全采纳）'](c_cal))
rB = action_risk(POLICIES['B 对称阈值 0.70'](c_cal))
rC = action_risk(POLICIES['C 非对称门控'](c_cal))
rD = action_risk(POLICIES['D 完全忽略感知'](c_cal))
rC_bad = action_risk(POLICIES['C 非对称门控'](c_over))
rC_under = action_risk(POLICIES['C 非对称门控'](c_under))

# ① 能拒绝就有收益；② 非对称门控收益更大；③ 感知确实有价值
assert rB[2] < rA[2] and rC[2] < rB[2] and rA[2] < rD[2]
# ④ 非对称门控专打「超速」这个高代价方向
assert rC[0] < rA[0] * 0.35, '超速率应至少降 65%'
# ⑤ 失准让规则失效
assert rC_bad[2] > rC[2] * 1.2, '过自信让加权风险涨 20% 以上'
assert rC_bad[0] > rC[0] * 1.8, '过自信让超速率翻倍'
# ⑥ 欠自信换来的是过度保守（换一组代价权重结论就会翻）
assert rC_under[1] > rC[1] and rC_under[2] > rC[2]

print(f'不传置信度         → 加权风险 {rA[2]:.4f}，超速率 {rA[0]:.2%}')
print(f'对称阈值（校准）    → 加权风险 {rB[2]:.4f}（{rB[2]/rA[2]-1:+.0%}）')
print(f'**非对称门控（校准）** → 加权风险 {rC[2]:.4f}（{rC[2]/rA[2]-1:+.0%}），'
      f'超速率 {rC[0]:.2%}（**{rC[0]/rA[0]-1:+.0%}**）')
print(f'非对称门控（过自信） → 加权风险 {rC_bad[2]:.4f}（比校准差 {rC_bad[2]/rC[2]-1:+.0%}），'
      f'超速率 {rC_bad[0]:.2%}')
print('\n★ 「p≥0.90 才允许提速」这条规则的**全部效力**建立在「0.90 真的意味着 90%」上。')
print('  过自信的模型把一堆实际 60% 正确率的检测标成 0.92 —— 门槛形同虚设。')
print(f'\n⚠️  保守不是免费的：非对称门控把过度限速率从 {rA[1]:.1%} 抬到 {rC[1]:.1%}。')
print('    正确做法不是「宁可错杀」，而是**把 C_over 与 C_under 显式写下来，由代价推出门槛** ——')
print('    这样产品说「减速太频繁」时，你改的是一个有明确含义的代价参数，不是凭感觉挪阈值。')

## 6 · 幻觉：输入幻觉、输出幻觉与 grounding 校验

用一个**规则式的玩具 VLA**（决定性、可断言）来演示「合理化」与两种幻觉。

In [ ]:
def toy_vla(dets, use_confidence=True, conf_gate=0.65):
    """玩具 VLA：忠实地以输入为条件生成动作与解释，并显式引用 track_id。"""
    considered = [d for d in dets
                  if (not use_confidence) or d['conf_cls'] >= conf_gate]
    if not considered:
        return dict(action='KEEP', target_kph=None,
                    explanation='未获得足够可信的标志信息，保持当前策略。',
                    cited=[], hedged=True)
    d = min(considered, key=lambda x: x['x_fwd_m'])     # 最近的那块牌子
    if d['cls'] == 'no_entry':
        return dict(action='LANE_CHANGE_RIGHT', target_kph=40,
                    explanation=(f"前方 {d['x_fwd_m']:.0f} m 检测到禁止驶入标志"
                                 f"(track#{d['track_id']})，本车道不可通行；"
                                 f"右侧车道空闲，建议提前变道并减速至 40 km/h 后并线。"),
                    cited=[d['track_id']], hedged=False)
    if d['cls'] == 'speed_limit':
        return dict(action='SET_SPEED', target_kph=d['value_kph'],
                    explanation=(f"依据 track#{d['track_id']} 限速标志，"
                                 f"设定目标车速 {d['value_kph']} km/h。"),
                    cited=[d['track_id']], hedged=False)
    return dict(action='KEEP', target_kph=None, explanation='无相关标志。',
                cited=[], hedged=True)

# —— 输入幻觉：一块**不存在的**禁止驶入（其实是广告牌），置信度 0.62 ——
FP = dict(track_id=9001, cls='no_entry', value_kph=None, conf_det=0.71, conf_cls=0.62,
          x_fwd_m=62.0, y_left_m=-2.4, sigma_dist_m=4.5, lane_assoc='ego',
          n_obs=1, state='tentative')
REAL = dict(DET)

out_naive = toy_vla([REAL, FP], use_confidence=False)
out_gated = toy_vla([REAL, FP], use_confidence=True, conf_gate=0.65)
print('【不传/不用置信度】')
print(' 动作:', out_naive['action'], '| 目标车速:', out_naive['target_kph'])
print(' 解释:', out_naive['explanation'])
print(' 表达了不确定吗:', out_naive['hedged'])
print('\n【置信度门控 0.65】')
print(' 动作:', out_gated['action'], '| 目标车速:', out_gated['target_kph'])
print(' 解释:', out_gated['explanation'])

assert out_naive['action'] == 'LANE_CHANGE_RIGHT' and not out_naive['hedged']
assert out_gated['action'] == 'SET_SPEED' and out_gated['target_kph'] == 60
print('\n⚠️  输出**语言流畅、逻辑自洽、完全错误**，而且没有任何一处表达「我不确定」。')
print('    这是 silent failure —— 比崩溃危险得多，因为没有信号触发兜底。')
print('✅ 一个 0.65 的门控就把它挡住了（真牌 0.71 通过、误检 0.62 被拦），')
print('   而门控能存在的前提，就是接口传了 conf_cls。')

In [ ]:
# —— 输出幻觉：VLA 引用了输入里根本不存在的 track_id ——
def grounding_check(vla_out, dets):
    """输出侧校验：引用的每个 track_id 必须存在于本帧输入。十几行、零延迟。"""
    valid = {d['track_id'] for d in dets}
    bad = [t for t in vla_out['cited'] if t not in valid]
    # 解释文本里出现的 track#NNNN 也要能对上
    mentioned = [int(x) for x in re.findall(r'track#(\d+)', vla_out['explanation'])]
    bad += [t for t in mentioned if t not in valid]
    return (len(bad) == 0), sorted(set(bad))

good = toy_vla([REAL], use_confidence=True)
ok1, bad1 = grounding_check(good, [REAL])
hallu = dict(good)
hallu['cited'] = [7777]
hallu['explanation'] = '路口通常设有停车让行标志(track#7777)，建议完全停车。'
ok2, bad2 = grounding_check(hallu, [REAL])
print(f'正常输出   grounding: {ok1}  未对上的 ID: {bad1}')
print(f'编造的输出 grounding: {ok2}  未对上的 ID: {bad2}  → **整条输出作废，走兜底**')
assert ok1 and not ok2 and bad2 == [7777]

# —— 度量：沉默错误率（错了、且输出未表达不确定）——
def silent_error_rate(n=4000, fp_rate=0.15, use_confidence=True, gate=0.65, seed=11):
    r = np.random.default_rng(seed)
    n_err = n_silent = 0
    for _ in range(n):
        dets = [dict(REAL)]
        truth = 'SET_SPEED'
        if r.random() < fp_rate:                      # 注入一次误检
            fp = dict(FP); fp['conf_cls'] = float(r.uniform(0.25, 0.85))
            dets.append(fp)
        out = toy_vla(dets, use_confidence=use_confidence, conf_gate=gate)
        if out['action'] != truth:
            n_err += 1
            if not out['hedged']:
                n_silent += 1
    return n_err / n, n_silent / n

for label, uc in [('不用置信度', False), ('置信度门控 0.65', True)]:
    er, sr = silent_error_rate(use_confidence=uc)
    print(f'{label:<16s} 下游动作错误率 {er:>6.2%}   其中**沉默错误** {sr:>6.2%}')
er_off, _ = silent_error_rate(use_confidence=False)
er_on, _ = silent_error_rate(use_confidence=True)
assert er_on < er_off * 0.5, '置信度门控应把动作错误率至少砍一半'
print('\n✅ 两类幻觉要用**不同手段**：输入幻觉靠置信度门控 + 代价不对称；')
print('   输出幻觉靠 grounding 校验（引用不到就整条作废）。')
print('★ 最该被追踪的单一指标是**沉默错误率**：错了、且输出里没有任何不确定信号。')

## 7 · prompt 模板的三种静默故障

位置错位 / 哨兵值被规则吃掉 / 数值格式漂移——**共同特征是不报错**。

In [ ]:
# —— 故障① 位置式解析 + 感知升级插入新字段 = 静默错位 ——
NEW_ORDER = FIELD_ORDER[:2] + ['src'] + FIELD_ORDER[2:]      # 感知 v1.4 插入了 src
DET_V14 = dict(DET); DET_V14['src'] = 'front_wide'

line_keyed_v14 = render_line(DET_V14, NEW_ORDER, keyed=True)
line_pos_v14 = render_line(DET_V14, NEW_ORDER, keyed=False)
pk = parse_keyed(line_keyed_v14)                 # 下游仍是 v1.3 的解析器
pp = parse_positional(line_pos_v14, FIELD_ORDER)  # 下游仍按旧顺序切

print('感知升级插入 src 字段后，下游仍用旧解析器：')
print(f'  键值式  x_fwd_m = {pk["x_fwd_m"]}   value_kph = {pk["value_kph"]}   ✅ 正确')
print(f'  位置式  x_fwd_m = {pp["x_fwd_m"]}   value_kph = {pp["value_kph"]}   ❌ **静默错位**')
assert pk['x_fwd_m'] == 82.4 and pk['value_kph'] == 60
assert pp['x_fwd_m'] != '82.4' and pp['value_kph'] != '60'
print(f'\n  位置式把**置信度读成了距离**（{pp["x_fwd_m"]}），还不报错。')
print(f'  token 成本：位置式 {token_est(line_pos_v14)} < 键值式 {token_est(line_keyed_v14)} ——')
print('  **省下的 token 换来的是一整类静默故障。不值。**')

In [ ]:
# —— 故障② 哨兵值 vs 显式 null ——
DET_NO_DIST = dict(DET); DET_NO_DIST['x_fwd_m'] = None; DET_NO_DIST['sigma_dist_m'] = None
line_null = render_line(DET_NO_DIST, null='null')
line_sentinel = render_line(DET_NO_DIST, null='-1')

def downstream_rule(parsed):
    """下游的一条常见规则：忽略车后目标（x<0）；距离未知则走保守分支。"""
    x = parsed['x_fwd_m']
    if x is None:
        return 'CONSERVATIVE(距离未知→保守)'
    return 'DROP(车后目标)' if float(x) < 0 else 'USE'

r_null = downstream_rule(parse_keyed(line_null))
r_sent = downstream_rule(parse_keyed(line_sentinel))
print(f'  缺失用 null  → {r_null}')
print(f'  缺失用 -1    → {r_sent}   ❌ **一个有效检测被静默丢掉，且没有任何日志**')
assert r_null.startswith('CONSERVATIVE') and r_sent.startswith('DROP')

# —— 故障③ 数值格式漂移 ——
print('\n数值格式漂移（训练时 60，推理时变成 60.0 / 6.0e+01）：')
for s in ['60', '60.0', '6.0e+01']:
    try:
        v = int(s); note = '✅ 解析成功'
    except ValueError:
        v = None; note = '❌ int() 抛异常 —— **fail loud，其实是好事**'
    print(f'  int({s!r}) -> {v}   token 数 {token_est(s)}   {note}')

# —— 自然语言 vs 紧凑 schema 的 token 账 ——
NL = ('There is a speed limit sign with track id 4127 detected ahead. The detection '
      'confidence is 0.93 and the classification confidence is 0.71. It indicates a '
      'speed limit of 60 kph. It is located 82.4 meters in front of the ego vehicle and '
      '3.1 meters to the right, with a distance uncertainty of 6.9 meters. It is '
      'associated with the ego lane, has been observed 3 times, and its state is tentative.')
t_keyed, t_nl = token_est(render_line(DET)), token_est(NL)
print(f'\n同样的信息：紧凑键值 {t_keyed} token vs 自然语言 {t_nl} token '
      f'→ **贵 {t_nl/t_keyed:.1f} 倍**，且无法严格解析与校验')
assert t_nl > 2 * t_keyed
print('\n✅ 三条铁律：①键值不要位置 ②字段顺序固定并写进 schema_version ③缺失值用显式 null')
print('★ **prompt 模板不是 UI，是 ABI** —— 它恰好长得像人话，而这正是它最危险的地方。')

## ✏️ 练习 1：完整 prompt 构建（稳定排序 + 截断标记 + 空场景）

实现 `build_prompt(dets, ego, max_tokens)`：
- 头部：`schema=tsr/1.3|ego_speed_mps=27.8`
- **稳定排序**：先按 `LANE_PRIORITY[lane_assoc]`，再按 `x_fwd_m` 升序（同值时按 `track_id`）
- 逐条加入 `render_line(d)`，用 `token_est` 估算，**并为尾部标记预留 `TAIL_TOKENS`**
- 装不下的丢弃并计数；空场景写 `signs=[]|n=0`
- 尾部固定为 `truncated=<true|false>|n_dropped=<M>`
- 各部分用 `\n` 连接

In [ ]:
# LANE_PRIORITY / TAIL_TOKENS / token_est 已在前面定义好，直接用

def build_prompt(dets, ego, max_tokens=500):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
def _mk(tid, lane, x, cls='speed_limit', v=60):
    d = dict(DET); d.update(track_id=tid, lane_assoc=lane, x_fwd_m=x, cls=cls, value_kph=v)
    return d

EGO = {'speed_mps': 27.8}
DETS = [_mk(11, 'service', 30.0), _mk(12, 'ego', 82.4), _mk(13, 'ego', 41.0),
        _mk(14, 'opposite', 25.0), _mk(15, 'left', 60.0), _mk(16, 'ego', 120.0)]

p_full = build_prompt(DETS, EGO, max_tokens=500)
ids = [int(m) for m in re.findall(r'track_id=(\d+)', p_full)]
assert ids == [13, 12, 16, 15, 14, 11], ids       # ego(按距离) -> left -> opposite/service
assert 'truncated=false|n_dropped=0' in p_full
assert p_full.startswith(f'schema={SCHEMA_VERSION}|ego_speed_mps=27.8')

# 顺序无关：打乱输入应得到**完全相同**的 prompt（否则 badcase 无法重放）
import random as _rd
sh = DETS[:]; _rd.Random(0).shuffle(sh)
assert build_prompt(sh, EGO, 500) == p_full, '排序必须稳定且与输入顺序无关'

p_tr = build_prompt(DETS, EGO, max_tokens=120)
kept = len(re.findall(r'track_id=', p_tr))
n_drop = int(re.search(r'n_dropped=(\d+)', p_tr).group(1))
assert 'truncated=true' in p_tr and kept + n_drop == len(DETS)
assert token_est(p_tr) <= 120, token_est(p_tr)
assert kept >= 1 and re.findall(r'track_id=(\d+)', p_tr)[0] == '13', '最该保留的是最近的本车道目标'

p_empty = build_prompt([], EGO, 500)
assert 'signs=[]|n=0' in p_empty and 'truncated=false' in p_empty
print(p_tr)
print(f'\n完整 {token_est(p_full)} token / 截断后 {token_est(p_tr)} token，丢弃 {n_drop} 个')
print('✅ 练习 1 通过：**排序稳定（可重放）+ 截断显式声明（模型知道自己看到的不全）+ ')
print('   空场景可表达（「我看了，确实没有」≠「什么都没写」）**')

## ✏️ 练习 2：后处理校准（温度搜索）

实现 `fit_temperature(conf_raw, correct, temps)`：对每个候选温度 $T$ 计算
$\mathrm{conf}' = \sigma(\mathrm{logit}(\mathrm{conf})/T)$ 的 ECE，返回 `(best_T, best_ece)`。
注意先把 `conf` 裁剪到 $[10^{-6}, 1-10^{-6}]$ 再取 logit。

In [ ]:
def fit_temperature(conf_raw, correct, temps):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
TEMPS = np.exp(np.linspace(np.log(0.2), np.log(6.0), 61))
raw_over = reported_conf(0.3)                    # 过自信：需要 T≈1/0.3≈3.33 才能校回来
bT, bE = fit_temperature(raw_over, is_correct, TEMPS)
print(f'过自信输入: 原始 ECE={ece(raw_over, is_correct):.4f} -> 校准后 ECE={bE:.4f}  (T*={bT:.2f})')
assert 2.5 < bT < 4.5, bT
assert bE < 0.01 and bE < ece(raw_over, is_correct) / 5

raw_under = reported_conf(2.5)                   # 欠自信：需要 T≈0.4
bT2, bE2 = fit_temperature(raw_under, is_correct, TEMPS)
print(f'欠自信输入: 原始 ECE={ece(raw_under, is_correct):.4f} -> 校准后 ECE={bE2:.4f}  (T*={bT2:.2f})')
assert 0.3 < bT2 < 0.55, bT2

bT3, _ = fit_temperature(reported_conf(1.0), is_correct, TEMPS)
assert 0.85 < bT3 < 1.2, '本来就校准的，T* 应接近 1'

# 校准之后，固定贝叶斯阈值下的代价应当恢复到最优
cal = sigmoid(logit(np.clip(raw_over, 1e-6, 1 - 1e-6)) / bT)
print(f'\n固定 τ*={TAU_BAYES} 下的期望代价: 未校准 {expected_cost(raw_over, TAU_BAYES):.4f} '
      f'-> 校准后 {expected_cost(cal, TAU_BAYES):.4f}')
assert expected_cost(cal, TAU_BAYES) < expected_cost(raw_over, TAU_BAYES) * 0.85
print('✅ 练习 2 通过：温度缩放只有**一个参数**，用几千条带标注的验证样本就能拟合，')
print('   却能把「概率可以被当作概率来算」这件事恢复回来 —— 性价比极高的一步。')

## ✏️ 练习 3：从代价推导门槛

实现 `bayes_threshold(c_wrong, c_rej)` 与
`asymmetric_apply(L_det, L_prior, conf, tau_up, tau_down)`：
提高限速（`L_det > L_prior`）要求 `conf >= tau_up`，否则只要 `conf >= tau_down`；
不满足则退回 `L_prior`。

In [ ]:
def bayes_threshold(c_wrong, c_rej):
    # TODO: p* = 1 - c_rej/c_wrong，裁剪到 [0, 1]
    raise NotImplementedError

def asymmetric_apply(L_det, L_prior, conf, tau_up, tau_down):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测（先手算）——
assert abs(bayes_threshold(10.0, 1.5) - 0.85) < 1e-12
assert abs(bayes_threshold(10.0, 3.0) - 0.70) < 1e-12
assert bayes_threshold(2.0, 5.0) == 0.0, '拒绝比犯错还贵 -> 永远采纳'
assert abs(bayes_threshold(100.0, 1.0) - 0.99) < 1e-12, '代价越不对称，门槛越高'

d_ = np.array([120, 60, 120, 60]); p_ = np.array([100, 100, 100, 100])
c_ = np.array([0.95, 0.50, 0.50, 0.20])
got = asymmetric_apply(d_, p_, c_, tau_up=0.90, tau_down=0.30)
assert got.tolist() == [120, 60, 100, 100], got.tolist()
#   ↑ 提速+高置信=采纳 / 降速+中置信=采纳 / 提速+中置信=拒绝 / 降速+极低置信=拒绝

# 用它复现主实验
tau_up, tau_down = 0.90, 0.30
ap = asymmetric_apply(L_det, L_prior, reported_conf(1.0), tau_up, tau_down)
o, u, r = action_risk(ap)
print(f'非对称门控(校准): 超速 {o:.2%}  过度限速 {u:.2%}  加权风险 {r:.4f}')
assert abs(r - rC[2]) < 1e-9

print(f"\n{'C_wrong':>8s} {'C_rej':>7s} {'p*':>7s} {'超速率':>8s} {'过度限速率':>11s} {'加权风险':>9s}")
for cw, cr in [(10.0, 5.0), (10.0, 3.0), (10.0, 1.5), (20.0, 1.0)]:
    t = bayes_threshold(cw, cr)
    a2 = asymmetric_apply(L_det, L_prior, reported_conf(1.0), tau_up=t, tau_down=0.30)
    o2, u2, r2 = action_risk(a2)
    print(f'{cw:>8.1f} {cr:>7.1f} {t:>7.2f} {o2:>8.2%} {u2:>11.2%} {r2:>9.4f}')
print('\n✅ 练习 3 通过：**门槛是代价的函数，不是调参得来的**。')
print('   产品说「减速太频繁」时，你改的是 C_rej 这个有明确含义的数，而不是凭感觉挪阈值。')

## ✏️ 练习 4：接口契约校验器

实现 `validate_message(msg)`，返回 `(ok, sorted(errors))`。检查项：
`schema_version` 在 `SUPPORTED_VERSIONS` 里；每条 sign 的 `REQUIRED` 字段都存在且非 `None`；
`conf_det`/`conf_cls` ∈ [0,1]；`x_fwd_m` ∈ [0, 300]；`sigma_dist_m` ≥ 0；
`lane_assoc` ∈ `LANE_PRIORITY`。错误串格式：`f'sign[{i}].{field}: {reason}'`，版本错误为 `'schema_version: unsupported'`。

In [ ]:
SUPPORTED_VERSIONS = {'tsr/1.2', 'tsr/1.3'}
REQUIRED = ['track_id', 'cls', 'conf_det', 'conf_cls', 'x_fwd_m', 'lane_assoc']

def validate_message(msg):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
good = {'schema_version': 'tsr/1.3', 'signs': [dict(DET)]}
ok, errs = validate_message(good)
assert ok and errs == [], errs

bad_sign = dict(DET)
bad_sign.update(conf_cls=1.4, x_fwd_m=-1.0, lane_assoc='ego_lane')
del bad_sign['track_id']
bad = {'schema_version': 'tsr/2.0', 'signs': [dict(DET), bad_sign]}
ok2, errs2 = validate_message(bad)
assert not ok2
assert 'schema_version: unsupported' in errs2
assert any(e.startswith('sign[1].track_id') for e in errs2)
assert any(e.startswith('sign[1].conf_cls') for e in errs2)
assert any(e.startswith('sign[1].x_fwd_m') for e in errs2)
assert any(e.startswith('sign[1].lane_assoc') for e in errs2)
assert not any(e.startswith('sign[0]') for e in errs2), '第 0 条是合法的，不应报错'
print('检出的问题:')
for e in errs2:
    print('  -', e)
print(f'\n共 {len(errs2)} 项')
assert len(errs2) == 5
print('\n✅ 练习 4 通过：**契约校验器要放在接口的消费侧，且必须 fail loud**。')
print('   它和第 3 节的坐标系断言测试一起，构成「把约定变成可执行检查」的最小集合。')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def build_prompt(dets, ego, max_tokens=500):
    ordered = sorted(dets, key=lambda d: (LANE_PRIORITY.get(d.get('lane_assoc'), 9),
                                          d['x_fwd_m'], d['track_id']))
    header = f"schema={SCHEMA_VERSION}|ego_speed_mps={ego['speed_mps']:.1f}"
    kept, dropped = [], 0
    for i, d in enumerate(ordered):
        cand = kept + [render_line(d)]
        if token_est('\n'.join([header] + cand)) + TAIL_TOKENS > max_tokens:
            dropped = len(ordered) - i
            break
        kept = cand
    body = '\n'.join(kept) if kept else 'signs=[]|n=0'
    tail = f"truncated={'true' if dropped else 'false'}|n_dropped={dropped}"
    return '\n'.join([header, body, tail])

In [ ]:
# 练习 2 参考答案
def fit_temperature(conf_raw, correct, temps):
    z = logit(np.clip(np.asarray(conf_raw, dtype=float), 1e-6, 1 - 1e-6))
    best_T, best_e = None, float('inf')
    for T in temps:
        e = ece(sigmoid(z / T), correct)
        if e < best_e:
            best_T, best_e = float(T), e
    return best_T, best_e

In [ ]:
# 练习 3 参考答案
def bayes_threshold(c_wrong, c_rej):
    return float(np.clip(1.0 - c_rej / c_wrong, 0.0, 1.0))

def asymmetric_apply(L_det, L_prior, conf, tau_up, tau_down):
    raising = np.asarray(L_det) > np.asarray(L_prior)
    accept = np.where(raising, np.asarray(conf) >= tau_up, np.asarray(conf) >= tau_down)
    return np.where(accept, L_det, L_prior)

In [ ]:
# 练习 4 参考答案
def validate_message(msg):
    errs = []
    if msg.get('schema_version') not in SUPPORTED_VERSIONS:
        errs.append('schema_version: unsupported')
    for i, s in enumerate(msg.get('signs', [])):
        for f in REQUIRED:
            if s.get(f) is None:
                errs.append(f'sign[{i}].{f}: missing')
        for f in ('conf_det', 'conf_cls'):
            v = s.get(f)
            if v is not None and not (0.0 <= v <= 1.0):
                errs.append(f'sign[{i}].{f}: out of [0,1] ({v})')
        x = s.get('x_fwd_m')
        if x is not None and not (0.0 <= x <= 300.0):
            errs.append(f'sign[{i}].x_fwd_m: out of [0,300] ({x})')
        sd = s.get('sigma_dist_m')
        if sd is not None and sd < 0:
            errs.append(f'sign[{i}].sigma_dist_m: negative ({sd})')
        la = s.get('lane_assoc')
        if la is not None and la not in LANE_PRIORITY:
            errs.append(f'sign[{i}].lane_assoc: unknown enum ({la})')
    return (len(errs) == 0), sorted(errs)

---
## 🧪 真实工程胶囊：TSR → VLA 接口规范与评审检查单

In [ ]:
RECIPE = r'''
# ══════════════════════════════════════════════════════════════════════
# TSR → VLA 感知输出接口：规范 + 评审检查单
# （可直接作为接口评审会的议程；每一条都对应一类真实故障）
# ══════════════════════════════════════════════════════════════════════

# ── ① 先定融合层次，再谈字段 ────────────────────────────────────────
#   判据：**这个信息出错以后，需不需要事后说清楚？**
#     需要（限速值、禁行、停车让行）-> 符号级，进日志、可审计
#     不需要（稠密几何、上下文）    -> 中间表示级 / BEV query，token 恒定
#   量产形态几乎总是混合的，两条通路互为交叉校验。

# ── ② schema（每个字段都要能回答「拿掉它，下游哪条判断做不了」）────
#   schema_version   接口演进的唯一抓手；进 prompt 第一行 + 进模型卡
#   t_capture_ms     数据新鲜度；配 ttl_ms 做看门狗
#   track_id         **跨帧确认、迟滞、状态机的前提**；没它 10 帧 = 10 块牌子
#   cls / cls_id     语义；同时是单目距离估计的**尺寸先验**
#   value_kph        限速值**独立成字段**（别编进类别名，会让类别数爆炸）
#   cls_top2         第二候选 + 概率 -> 下游才能表达「60 还是 80 说不准」
#   conf_det/conf_cls **分开传，不要提前融合**（下游用法不同）
#   x_fwd_m/y_left_m 自车坐标，**单位写进字段名**；不要传图像像素
#   sigma_dist_m     σ_Z ∝ Z²；12px 的框 -> 83m ± 6.9m
#   bbox_wh_px       距离估计输入 + 免费的可靠性代理
#   lane_assoc       **管不管我**；感知有车道线有能力算，下游只能猜
#   n_obs / state    时序一致性，与置信度**正交**
#   src / ttl_ms     多相机仲裁 / 有效期

# ── ③ 置信度（本接口的核心，单列一条）────────────────────────────
#   · 必须传。不传 = **永久删除下游做风险决策的能力**
#   · 必须校准。用几千条验证样本拟合一个温度参数即可（1 个参数！）
#       T* = argmin_T ECE(sigmoid(logit(conf)/T))
#   · 阈值从代价推导，不是调出来的：p* = 1 - C_rej / C_wrong
#   · **非对称门控**：往安全方向低门槛、往风险方向高门槛
#       raise_limit  -> require conf >= 0.90
#       lower_limit  -> require conf >= 0.30
#     实测：超速率 11.4% -> 3.1%（-72%），加权风险 -44%
#   · 校准漂移要监控：夜间/雨雾/新区域上会失效 -> **分场景各校一个温度**

# ── ④ prompt 模板 = ABI，不是 UI ───────────────────────────────────
#   ① 键值不要位置（位置式在插字段时**静默错位**）
#   ② 字段顺序固定；改顺序 = 升版本 = 重新评测
#   ③ 缺失值用显式 null；**绝不用 -1 这类哨兵**（会被下游规则静默吃掉）
#   ④ 数值格式定死（60.0 vs 60 是不同 token 序列 —— 文本层的训练/部署一致性）
#   ⑤ 排序规则确定（lane 优先级 -> 距离 -> track_id），否则同帧两次渲染不同
#   ⑥ 截断要显式声明：truncated=true|n_dropped=7
#   ⑦ 空场景要能表达：signs=[]|n=0（「我看了确实没有」≠「什么都没写」）

# ── ⑤ 必须实现的四个检查（加起来不到 100 行）─────────────────────
#   (a) round-trip:  parse(render(d)) == d，逐字段
#   (b) 坐标系断言:  放一个 y_left=+5 的合成目标，检查下游认为它在左边
#   (c) 契约校验器:  必填 / 范围 / 枚举 / 版本；**fail loud，绝不「尽力猜」**
#   (d) grounding:   VLA 输出引用的 track_id 必须存在于本帧输入，否则整条作废
#   把一份「金标准 prompt」纳入回归测试，任何渲染改动都要 diff 它。

# ── ⑥ 必须监控的指标 ───────────────────────────────────────────────
#   · ECE（总体 + 分场景：夜间/雨雾/距离分桶）
#   · **沉默错误率** = 错了、且输出未表达不确定 的比例   <- 最重要的单一指标
#   · 幻觉率 = 输出引用了输入里不存在的 track_id 的比例
#   · 下游动作错误率，按「超速 / 过度限速」**分方向**统计（代价不对称）
#   · 接口层：解析失败率、契约校验失败率、截断触发率、schema 版本分布

# ── ⑦ 红线 ─────────────────────────────────────────────────────────
#   · 训练数据必须包含**带噪感知输入 + 保守专家动作**的样本，
#     否则模型不知道该怎么用 conf 这个数字，前面所有门控都只是外挂规则
#   · 解释输出必须带上依赖的证据与置信度
#     （「依据 track#4127，置信 0.62，距离 82±7 m」）——
#     **流畅的解释会制造它并不配得的信任**
#   · 感知不要越界替下游做「该不该生效」的判断（涉及导航与法规优先级）
#     检验：如果这个判断错了，下游有没有信息把它纠正回来？没有就不该由感知定
'''
print(RECIPE)
for tok in ['schema_version', 'track_id', 'lane_assoc', 'sigma_dist_m',
            'p* = 1 - C_rej / C_wrong', '非对称门控', 'round-trip', 'grounding',
            '沉默错误率', 'ABI', 'null']:
    assert tok in RECIPE, tok
print('✅ 检查单覆盖：融合层次 / schema 逐字段 / 置信度与校准 / 模板铁律 / '
      '四个检查 / 监控指标 / 红线')

### 小结

- **接口设计的第一个问题不是「传什么字段」，而是「在哪一层传」**。三个层次的判据是
  **「这个信息出错以后需不需要事后说清楚」**——需要就必须走符号级（可审计），
  不需要就走中间表示级（token 恒定、信息最全）。量产形态是混合的。
- **符号级的 bit/token 效率只有特征级的 1/1000**（2.0 vs 2048 bit/token）。
  付出这个代价换的不是信息量，是**可解释性与可归因性**。
- **每个字段都要能回答「拿掉它，下游哪条判断做不了」**。最易漏又最致命的是
  `track_id`（跨帧确认）、`lane_assoc`（管不管我）、`sigma_dist_m`（12 px → 83 m ± 6.9 m，
  且 $\sigma_Z \propto Z^2$）、`n_obs`/`state`（与置信度**正交**的时序一致性）。
- **不传置信度不是少传一个字段，是永久删除了下游做风险决策的能力。**
  本例中它的期望代价比最优高 **97%**。而阈值应当从代价推导：$p^\star = 1 - C_{\text{rej}}/C_{\text{wrong}}$。
- **校准不是为了改善排序**（AUC 完全不变，重调阈值后可达代价也相同）——
  **是为了让「概率」这个数能被当作概率来算**。它在三种情况下不可替代：
  固定/共享阈值、用概率做算术、多源融合。温度缩放只有一个参数，性价比极高。
- **非对称门控**把代价不对称写进规则（提速要 $p\ge0.90$、降速只要 $p\ge0.30$），
  超速率 11.4% → 3.1%（**−72%**）；但**换成未校准的输入，这个收益直接打对折**。
- **VLA 会「合理化」错误输入**，输出流畅自洽且**不带任何不确定信号**——silent failure。
  输入幻觉靠置信度门控，输出幻觉靠 **grounding 校验**（引用不到的 `track_id` → 整条作废）。
  最该追踪的单一指标是**沉默错误率**。
- **prompt 模板是 ABI 不是 UI**：键值不要位置、顺序进版本、缺失用显式 `null` 而非哨兵。
  这三类故障的共同特征是**静默**，而排查静默故障的成本是排查崩溃的几十倍。

下一站：**模块 04 · 交通规则的指令跟随与约束化** ——
把本模块传下去的标志语义，翻译成有作用域、有生命周期、有优先级的可执行约束。